In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
from ioutils import FileIO
from gate import IOStore
from pandas import Series
from numpy import ceil
io        = FileIO()
ios       = IOStore()
mdbio     = ios.get("RateYourMusic")
refData   = mdbio.data.getSummaryRefData()
rymRefs   = refData.apply(lambda x: x.replace("https://rateyourmusic.com", "").replace("-", "_") if isinstance(x,str) else None)
rymRefs   = {v: k for k,v in rymRefs[rymRefs.notna()].to_dict().items()}

def fixFloat(x):
    try:
        retval = int(x)
    except:
        retval = x
    return retval

def removeKnownRefs(refs):
    if isinstance(refs,list):
        refs = [ref for ref in refs if rymRefs.get(ref.replace("-", "_")) is None]        
        retval = []
        for ref in refs:
            refname = ref.replace("-", "_")
            if masterRefs.get(refname) is not None:
                continue
            masterRefs[refname] = True
            retval.append(ref)
        return retval
    return []

In [ ]:
from pandas import concat
urlsToGet = concat([io.get("urlsToGet2.1.p"), io.get("urlsToGet1.1.p")])
urlsToGet = urlsToGet.sort_values(by="Rank", ascending=True)
print(f"# Found {len(rymRefs)} Known RateYourMusic Refs")
print(f"# Found {urlsToGet.shape[0]} Artists URLs To Get")
masterRefs = {}
urlsToGet["ToGet"] = urlsToGet["URefs"].apply(lambda x: removeKnownRefs(x))
done = urlsToGet[urlsToGet["ToGet"].map(len) == 0]
print(f"# Found {done.shape[0]} Artists With Known URLs")
refsToGet = urlsToGet[(urlsToGet["ToGet"].map(len) > 0)] # & urlsToGet["Rank"] > 132036]
print(f"# Found {refsToGet.shape[0]} Artists URLs To Download")

# Found 2773 Artists URLs To Get
# Found 77 Artists With Known URLs
# Found 2696 Artists URLs To Download

In [ ]:
head = 8
hset = 15
N    = refsToGet.shape[0]
nT   = int(ceil(N/hset))
for i,(_,row) in enumerate(refsToGet[((head-1)*hset):((head)*hset)].iterrows()):
    refs   = row["URefs"]
    name   = row["ArtistName"]
    rank   = fixFloat(row["Rank"])
    #if rank < 357559: continue
    counts = fixFloat(row["Counts"])
    n      = (head-1)*hset+i+1
    for ref in refs:
        url    = "https://rateyourmusic.com{0}".format(ref)
        print(f"{head: >3} / {nT: <3} | {n: >5} / {N: <4} |  {name: <40}{rank: <10}{counts: <4} | {url}")

In [ ]:
132036

In [ ]:
urlsToGet = io.get("urlsToGet3.p")
urlsToGet

In [ ]:
urlsToGet = io.get("urlsToGet.p")

# Parse

In [ ]:
from lib import rateyourmusic
rateyourmusic.moveLocalFiles()
rateyourmusic.removeLocalFiles()

from utils import PoolIO
pio = PoolIO("RateYourMusic")
pio.parse()
pio.meta()
pio.sum()
pio.search()

In [ ]:

x